# Notebook 3: Portfolio Optimisation
**AdaptiveBeta — AI-Powered Portfolio Optimisation**  
**Author:** Kunal | M.Tech AI & ML, Symbiosis Institute of Technology, Pune

This notebook:
1. Calibrates the beta volatility rebalancing threshold from training data
2. Implements the 3-signal decision tree (VIX override → beta threshold → hold)
3. Tests all 3 portfolio optimisers: max-Sharpe MVO, min-variance, risk parity
4. Validates the optimisers on recent out-of-sample dates

**The Threshold Trigger — Key Innovation:**  
Instead of rebalancing on a fixed schedule, we only rebalance when predicted
beta volatility crosses the 75th percentile of the training distribution.

In [ ]:
import sys, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/AI_Finance_Project'

REPO = '/content/drive/MyDrive/AI_Finance_Project/repo'
if os.path.exists(REPO):
    sys.path.insert(0, REPO)

print('Drive mounted.')

## 3.1 — Load Data & Calibrate Threshold

In [ ]:
TRAIN_END  = '2021-12-31'
TEST_START = '2022-01-01'
VIX_THRESHOLD = 25.0
THRESHOLD_QUANTILE = 0.75

# Load 60d betavol panel
betavol_60 = pd.read_csv(f'{ROOT}/features/betavol_60d.csv',
                          parse_dates=['date']).set_index('date')

# Portfolio-level betavol = equal-weighted average across all stocks
portfolio_betavol = betavol_60.mean(axis=1).dropna()

# Threshold = 75th percentile of TRAINING period only
train_betavol = portfolio_betavol[portfolio_betavol.index <= TRAIN_END]
threshold = float(train_betavol.quantile(THRESHOLD_QUANTILE))

print(f'Portfolio betavol statistics (training period):')
print(f'  Min:     {train_betavol.min():.4f}')
print(f'  Median:  {train_betavol.median():.4f}')
print(f'  75th pct: {threshold:.4f}  ← REBALANCE THRESHOLD')
print(f'  Max:     {train_betavol.max():.4f}')
print(f'\nVIX override threshold: {VIX_THRESHOLD}')

# % of days that would trigger rebalance in training
trigger_pct = (train_betavol > threshold).mean() * 100
print(f'\nIn training: {trigger_pct:.1f}% of days would trigger rebalance')

In [ ]:
# Visualise threshold
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Time series
portfolio_betavol.plot(ax=ax1, color='#1D9E75', lw=1, label='Portfolio BetaVol')
ax1.axhline(threshold, color='#EF9F27', ls='--', lw=1.5, label=f'Threshold={threshold:.4f}')
ax1.axvline(pd.Timestamp(TEST_START), color='#7F77DD', ls=':', lw=1.2, label='Train/Test split')
ax1.fill_between(portfolio_betavol.index, threshold, portfolio_betavol,
                  where=portfolio_betavol > threshold, alpha=0.3, color='#EF9F27', label='Rebalance zone')
ax1.set_title('Portfolio Beta Volatility & Rebalance Threshold')
ax1.legend(fontsize=9)

# Distribution
ax2.hist(train_betavol.dropna(), bins=50, color='#1D9E75', alpha=0.7, label='Training betavol')
ax2.axvline(threshold, color='#EF9F27', ls='--', lw=2, label=f'75th pct = {threshold:.4f}')
ax2.set_title('Distribution of Portfolio Beta Volatility (Training)')
ax2.set_xlabel('Beta Volatility')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{ROOT}/results/threshold_calibration.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.2 — Rebalance Signal Logic

In [ ]:
def get_rebalance_signal(predicted_betavol: float, vix_level: float) -> str:
    """
    Priority order:
    1. VIX > 25 → MIN_VARIANCE (market stress, go defensive)
    2. predicted_betavol > threshold → REBALANCE
    3. Otherwise → HOLD
    """
    if vix_level > VIX_THRESHOLD:
        return 'MIN_VARIANCE'
    elif predicted_betavol > threshold:
        return 'REBALANCE'
    else:
        return 'HOLD'

def get_portfolio_mode(signal: str, regime: str) -> str:
    """Route signal + regime to specific optimiser."""
    if signal == 'MIN_VARIANCE':
        return 'min_variance'
    elif signal == 'REBALANCE':
        if regime == 'bull':        return 'max_sharpe_beta_constrained'
        elif regime == 'bear':      return 'min_variance'
        else:                       return 'risk_parity'
    else:
        return 'hold'

# Test the signal logic
print('Signal logic examples:')
test_cases = [
    (threshold * 0.8, 20, 'bull'),   # below threshold, normal VIX
    (threshold * 1.2, 20, 'bull'),   # above threshold, bull regime
    (threshold * 1.2, 20, 'bear'),   # above threshold, bear regime
    (threshold * 0.5, 28, 'bull'),   # VIX override
    (threshold * 1.5, 30, 'transition'),  # both triggers
]

print(f'{"Pred BetaVol":>15} | {"VIX":>5} | {"Regime":>12} | {"Signal":>15} | {"Mode":>30}')
print('-' * 90)
for pred_bv, vix_lvl, regime in test_cases:
    sig  = get_rebalance_signal(pred_bv, vix_lvl)
    mode = get_portfolio_mode(sig, regime)
    print(f'{pred_bv:>15.4f} | {vix_lvl:>5.1f} | {regime:>12} | {sig:>15} | {mode:>30}')

## 3.3 — Portfolio Optimisers

In [ ]:
from pypfopt import EfficientFrontier, risk_models, expected_returns

def optimise_portfolio(
    prices_window: pd.DataFrame,
    current_betas: pd.Series,
    mode: str,
    beta_target: float = 0.85,
    max_weight: float = 0.15,
    risk_free_rate: float = 0.065,
) -> pd.Series:
    """
    Returns optimal weights as pd.Series indexed by ticker.

    mode: 'max_sharpe_beta_constrained' | 'min_variance' | 'risk_parity' | 'hold'
    """
    n = len(prices_window.columns)
    tickers = prices_window.columns.tolist()

    # Covariance (Ledoit-Wolf shrinkage for stability with 49 assets)
    S = risk_models.CovarianceShrinkage(prices_window).ledoit_wolf()

    if mode == 'max_sharpe_beta_constrained':
        mu = expected_returns.mean_historical_return(prices_window, frequency=252)
        betas_arr = current_betas.reindex(tickers).fillna(1.0).values

        ef = EfficientFrontier(mu, S)
        ef.add_constraint(lambda w: w >= 0)              # long only
        ef.add_constraint(lambda w: w <= max_weight)     # max 15%

        # Beta band: portfolio beta ∈ [beta_target - 0.2, beta_target]
        ef.add_constraint(lambda w: sum(w[i] * betas_arr[i] for i in range(n)) <= beta_target)
        ef.add_constraint(lambda w: sum(w[i] * betas_arr[i] for i in range(n)) >= beta_target - 0.2)

        try:
            ef.max_sharpe(risk_free_rate=risk_free_rate)
            return pd.Series(ef.clean_weights())
        except Exception as e:
            print(f'  Max Sharpe failed ({e}) → falling back to min_variance')
            mode = 'min_variance'

    if mode == 'min_variance':
        ef = EfficientFrontier(None, S)
        ef.add_constraint(lambda w: w >= 0)
        ef.add_constraint(lambda w: w <= max_weight)
        ef.min_volatility()
        return pd.Series(ef.clean_weights())

    if mode == 'risk_parity':
        try:
            import riskfolio as rp
            returns = np.log(prices_window / prices_window.shift(1)).dropna()
            port = rp.Portfolio(returns=returns)
            port.assets_stats(method_mu='hist', method_cov='ledoit')
            w = port.optimization(model='Classic', rm='MV', obj='Sharpe',
                                   rf=risk_free_rate, l=0, hist=True)
            if w is not None:
                return w['weights']
        except Exception as e:
            print(f'  Risk parity failed ({e}) → min_variance')
        # Fallback
        ef = EfficientFrontier(None, S)
        ef.add_constraint(lambda w: w >= 0)
        ef.add_constraint(lambda w: w <= max_weight)
        ef.min_volatility()
        return pd.Series(ef.clean_weights())

    # Equal weight fallback
    return pd.Series({t: 1/n for t in tickers})

print('Portfolio optimiser defined.')

## 3.4 — Test Optimisers on Recent Data

In [ ]:
# Load data for optimiser test
prices_df = pd.read_csv(f'{ROOT}/raw_data/stocks/all_stocks_prices.csv',
                         parse_dates=['date']).set_index('date').sort_index()

TICKERS = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS',
    'HINDUNILVR.NS','ITC.NS','SBIN.NS','BHARTIARTL.NS','KOTAKBANK.NS',
    'LT.NS','AXISBANK.NS','ASIANPAINT.NS','MARUTI.NS','BAJFINANCE.NS',
    'HCLTECH.NS','SUNPHARMA.NS','TITAN.NS','ULTRACEMCO.NS','NESTLEIND.NS',
    'WIPRO.NS','POWERGRID.NS','NTPC.NS','ONGC.NS','TECHM.NS',
    'JSWSTEEL.NS','TATASTEEL.NS','M&M.NS','ADANIENT.NS','ADANIPORTS.NS',
    'COALINDIA.NS','BAJAJFINSV.NS','HDFCLIFE.NS','SBILIFE.NS','DRREDDY.NS',
    'DIVISLAB.NS','CIPLA.NS','EICHERMOT.NS','HEROMOTOCO.NS','APOLLOHOSP.NS',
    'BAJAJ-AUTO.NS','BRITANNIA.NS','GRASIM.NS','INDUSINDBK.NS','TATACONSUM.NS',
    'UPL.NS','BPCL.NS','IOC.NS','HINDALCO.NS',
]

avail = [t for t in TICKERS if t in prices_df.columns]
prices = prices_df[avail]

# Use last 252 trading days of training data as price window
train_prices = prices.loc[:TRAIN_END].tail(252).dropna(axis=1, how='any')
test_tickers = train_prices.columns.tolist()

# Mock betas: use median values from beta panel
betas_60 = pd.read_csv(f'{ROOT}/features/beta60d.csv', parse_dates=['date']).set_index('date')
last_beta = betas_60.loc[:TRAIN_END].iloc[-1].reindex(test_tickers).fillna(1.0)

print(f'Test price window: {train_prices.shape}')
print(f'Available tickers: {len(test_tickers)}')

for mode in ['max_sharpe_beta_constrained', 'min_variance', 'risk_parity']:
    print(f'\nTesting mode: {mode}')
    try:
        w = optimise_portfolio(train_prices, last_beta, mode, beta_target=0.85)
        w_nonzero = w[w > 0.001]
        portfolio_beta = (w * last_beta.reindex(w.index).fillna(1.0)).sum()
        print(f'  Positions: {len(w_nonzero)} | Max weight: {w_nonzero.max():.3f} | '
              f'Portfolio beta: {portfolio_beta:.3f}')
        print(f'  Top 5: {w_nonzero.nlargest(5).to_dict()}')
    except Exception as e:
        print(f'  FAILED: {e}')

In [ ]:
# Save threshold config for use in Notebook 4
import json
config = {
    'betavol_threshold': threshold,
    'vix_threshold': VIX_THRESHOLD,
    'threshold_quantile': THRESHOLD_QUANTILE,
    'train_end': TRAIN_END,
    'test_start': TEST_START,
}
with open(f'{ROOT}/models/signal_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved: signal_config.json')
print(config)
print('\n✅ Notebook 3 complete — proceed to Notebook 4: Backtesting')